# Assignment 2: Sanskrit-to-English Neural Machine Translation
> **Course:** Natural Language Understanding  
> **Architecture:** Seq2Seq · BiLSTM Encoder · Bahdanau Attention · LSTM Decoder · Beam Search  
> **Evaluation:** BLEU (NLTK, no weights) · BERTScore F1 (rescale_with_baseline=True) · Efficiency

---
### Notebook Structure
1. Installation & Imports  
2. Configuration (all tuneable params in one place)  
3. Data Loading & Preprocessing  
4. Vocabulary  
5. Dataset & DataLoader  
6. Model: Encoder · Attention · Decoder · Seq2Seq  
7. Training  
8. Inference (Greedy + Beam Search)  
9. Evaluation: BLEU · BERTScore · Efficiency  
10. Submission CSV  
11. Translation Examples & Error Analysis

## 1. Installation

In [ ]:
# Install all required dependencies
!pip install torch torchtext sacrebleu bert-score nltk indic-nlp-library --quiet
!pip install pandas numpy matplotlib tqdm --quiet

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('All dependencies installed.')

## 2. Imports

In [ ]:
import os, time, math, random, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from bert_score import score as bert_score_fn

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 3. Configuration
> **All dataset paths and hyperparameters are centralised here.**  
> To run on a new dataset, update `DATA_CONFIG` only.

In [ ]:
# ─── DATA CONFIG (parameterised — update paths here for new datasets) ──────────
DATA_CONFIG = {
    # Directory containing all CSVs (override for Colab/Drive)
    'data_dir': './data',

    # File names (column names: Source_id, Sentence_sa / Sentence_en)
    'train_sa': 'train_sa.csv',
    'train_en': 'train_en.csv',
    'dev_sa'  : 'dev_sa.csv',
    'dev_en'  : 'dev_en.csv',
    'test_sa' : 'test_sa.csv',
    # test_en is not provided — we generate predictions for it

    # Column names (change if your CSVs use different headers)
    'id_col'  : 'Source_id',
    'sa_col'  : 'Sentence_sa',
    'en_col'  : 'Sentence_en',
}

# ─── MODEL HYPERPARAMETERS ──────────────────────────────────────────────────────
MODEL_CONFIG = {
    'embed_dim'     : 256,
    'hidden_dim'    : 512,    # per direction in encoder
    'n_layers'      : 2,
    'attn_dim'      : 256,
    'dropout'       : 0.3,
    'max_src_len'   : 100,    # truncate Sanskrit sequences longer than this
    'max_trg_len'   : 100,    # truncate English sequences longer than this
    'min_freq'      : 1,      # min token frequency to include in vocab
}

# ─── TRAINING HYPERPARAMETERS ───────────────────────────────────────────────────
TRAIN_CONFIG = {
    'epochs'               : 30,
    'batch_size'           : 64,
    'learning_rate'        : 3e-4,
    'clip_grad'            : 1.0,
    'teacher_forcing_ratio': 0.5,
    'lr_patience'          : 3,     # ReduceLROnPlateau patience
    'early_stop_patience'  : 6,     # stop if dev loss doesn't improve
    'checkpoint_path'      : 'best_model.pt',
}

# ─── INFERENCE CONFIG ───────────────────────────────────────────────────────────
INFER_CONFIG = {
    'beam_size'    : 5,
    'max_decode_len': 80,
    'length_penalty': 0.7,   # beam search length penalty alpha
}

# ─── SPECIAL TOKENS ─────────────────────────────────────────────────────────────
PAD_TOKEN = '<PAD>'   # index 0
SOS_TOKEN = '<SOS>'   # index 1
EOS_TOKEN = '<EOS>'   # index 2
UNK_TOKEN = '<UNK>'   # index 3

print('Configuration loaded.')
print(f"Data directory: {DATA_CONFIG['data_dir']}")

## 4. Data Loading & Preprocessing

In [ ]:
# ─── OPTIONAL: Mount Google Drive (uncomment in Colab) ─────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_CONFIG['data_dir'] = '/content/drive/MyDrive/sanskrit_nmt/data'


def load_parallel_data(
    data_dir: str,
    sa_file: str,
    en_file: Optional[str],
    id_col: str,
    sa_col: str,
    en_col: str,
) -> pd.DataFrame:
    """Load and align Sanskrit–English parallel CSV files by Source_id."""
    sa_path = Path(data_dir) / sa_file
    df_sa = pd.read_csv(sa_path, encoding='utf-8')
    df_sa = df_sa[[id_col, sa_col]].dropna()

    if en_file is None:
        # Test set: no English reference
        return df_sa.rename(columns={sa_col: 'sanskrit'})

    en_path = Path(data_dir) / en_file
    df_en = pd.read_csv(en_path, encoding='utf-8')
    df_en = df_en[[id_col, en_col]].dropna()

    df = pd.merge(df_sa, df_en, on=id_col)
    df = df.rename(columns={sa_col: 'sanskrit', en_col: 'english'})
    df = df.reset_index(drop=True)
    return df


def basic_tokenize(text: str) -> List[str]:
    """Whitespace + punctuation tokeniser — works for both Sanskrit and English."""
    text = str(text).strip()
    # Separate punctuation from words
    text = re.sub(r'([।॥\.\,\!\?\;\:\'\"\(\)\[\]\{\}])', r' \1 ', text)
    tokens = text.split()
    return [t for t in tokens if t]


def preprocess_df(
    df: pd.DataFrame,
    max_src_len: int,
    max_trg_len: int,
    has_target: bool = True,
) -> pd.DataFrame:
    """Tokenise and optionally filter long sequences."""
    df = df.copy()
    df['src_tokens'] = df['sanskrit'].apply(basic_tokenize)
    if has_target:
        df['trg_tokens'] = df['english'].apply(
            lambda x: basic_tokenize(str(x).lower())
        )
        # Filter extreme lengths (training only)
        mask = (
            df['src_tokens'].apply(len).between(1, max_src_len) &
            df['trg_tokens'].apply(len).between(1, max_trg_len)
        )
        df = df[mask].reset_index(drop=True)
    else:
        df['trg_tokens'] = [[] for _ in range(len(df))]
    return df


# Load all splits
cfg, mc, tc = DATA_CONFIG, MODEL_CONFIG, TRAIN_CONFIG

raw_train = load_parallel_data(cfg['data_dir'], cfg['train_sa'], cfg['train_en'],
                               cfg['id_col'], cfg['sa_col'], cfg['en_col'])
raw_dev   = load_parallel_data(cfg['data_dir'], cfg['dev_sa'],   cfg['dev_en'],
                               cfg['id_col'], cfg['sa_col'], cfg['en_col'])
raw_test  = load_parallel_data(cfg['data_dir'], cfg['test_sa'],  None,
                               cfg['id_col'], cfg['sa_col'], cfg['en_col'])

train_df = preprocess_df(raw_train, mc['max_src_len'], mc['max_trg_len'], has_target=True)
dev_df   = preprocess_df(raw_dev,   mc['max_src_len'], mc['max_trg_len'], has_target=True)
test_df  = preprocess_df(raw_test,  mc['max_src_len'], mc['max_trg_len'], has_target=False)

print(f'Train: {len(train_df):,} pairs | Dev: {len(dev_df):,} pairs | Test: {len(test_df):,} sentences')
print('\nSample training pair:')
print(f"  Sanskrit : {train_df['sanskrit'].iloc[0]}")
print(f"  English  : {train_df['english'].iloc[0]}")
print(f"  Src tokens: {train_df['src_tokens'].iloc[0]}")
print(f"  Trg tokens: {train_df['trg_tokens'].iloc[0]}")

## 5. Vocabulary

In [ ]:
class Vocabulary:
    """Token ↔ index mapping with special token support."""

    SPECIALS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.token2idx: Dict[str, int] = {}
        self.idx2token: Dict[int, str] = {}
        self._build_specials()

    def _build_specials(self):
        for i, tok in enumerate(self.SPECIALS):
            self.token2idx[tok] = i
            self.idx2token[i]   = tok

    def build(self, token_lists: List[List[str]]) -> 'Vocabulary':
        counter = Counter(tok for seq in token_lists for tok in seq)
        for tok, freq in counter.most_common():
            if freq >= self.min_freq and tok not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[tok] = idx
                self.idx2token[idx] = tok
        return self

    def encode(self, tokens: List[str], add_sos=False, add_eos=True) -> List[int]:
        ids = [self.token2idx.get(t, self.token2idx[UNK_TOKEN]) for t in tokens]
        if add_sos: ids = [self.token2idx[SOS_TOKEN]] + ids
        if add_eos: ids = ids + [self.token2idx[EOS_TOKEN]]
        return ids

    def decode(self, ids: List[int], strip_special=True) -> str:
        tokens = [self.idx2token.get(i, UNK_TOKEN) for i in ids]
        if strip_special:
            tokens = [t for t in tokens
                      if t not in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN)]
        return ' '.join(tokens)

    @property
    def pad_idx(self): return self.token2idx[PAD_TOKEN]

    @property
    def sos_idx(self): return self.token2idx[SOS_TOKEN]

    @property
    def eos_idx(self): return self.token2idx[EOS_TOKEN]

    def __len__(self): return len(self.token2idx)


# Build vocabularies from training data only (no data leakage)
src_vocab = Vocabulary(min_freq=mc['min_freq']).build(train_df['src_tokens'].tolist())
trg_vocab = Vocabulary(min_freq=mc['min_freq']).build(train_df['trg_tokens'].tolist())

print(f'Source vocab size (Sanskrit): {len(src_vocab):,}')
print(f'Target vocab size (English) : {len(trg_vocab):,}')

## 6. Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    """PyTorch Dataset for Sanskrit–English parallel data."""

    def __init__(
        self,
        df: pd.DataFrame,
        src_vocab: Vocabulary,
        trg_vocab: Vocabulary,
        has_target: bool = True,
    ):
        self.src_vocab   = src_vocab
        self.trg_vocab   = trg_vocab
        self.has_target  = has_target
        self.source_ids  = df[DATA_CONFIG['id_col']].tolist()

        self.src_encoded = [
            src_vocab.encode(row, add_sos=False, add_eos=True)
            for row in df['src_tokens']
        ]
        self.trg_encoded = [
            trg_vocab.encode(row, add_sos=True, add_eos=True)
            for row in df['trg_tokens']
        ] if has_target else [[] for _ in range(len(df))]

    def __len__(self): return len(self.src_encoded)

    def __getitem__(self, idx):
        return {
            'source_id': self.source_ids[idx],
            'src'      : torch.tensor(self.src_encoded[idx], dtype=torch.long),
            'trg'      : torch.tensor(self.trg_encoded[idx], dtype=torch.long)
                         if self.has_target else torch.tensor([], dtype=torch.long),
        }


def collate_fn(batch: List[dict]) -> dict:
    """Pad sequences within a batch to the same length."""
    source_ids = [b['source_id'] for b in batch]
    src_seqs   = [b['src'] for b in batch]
    trg_seqs   = [b['trg'] for b in batch]

    src_padded = pad_sequence(src_seqs, batch_first=True,
                              padding_value=src_vocab.pad_idx)
    has_trg = all(len(t) > 0 for t in trg_seqs)
    trg_padded = pad_sequence(trg_seqs, batch_first=True,
                              padding_value=trg_vocab.pad_idx) if has_trg else None
    return {'source_id': source_ids, 'src': src_padded, 'trg': trg_padded}


def make_dataloader(df, src_vocab, trg_vocab, batch_size,
                    shuffle=False, has_target=True) -> DataLoader:
    ds = TranslationDataset(df, src_vocab, trg_vocab, has_target=has_target)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=0, pin_memory=True)


train_loader = make_dataloader(train_df, src_vocab, trg_vocab,
                               tc['batch_size'], shuffle=True)
dev_loader   = make_dataloader(dev_df,   src_vocab, trg_vocab,
                               tc['batch_size'], shuffle=False)
test_loader  = make_dataloader(test_df,  src_vocab, trg_vocab,
                               tc['batch_size'], shuffle=False, has_target=False)

print(f'Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)} | Test batches: {len(test_loader)}')

## 7. Model Architecture

In [ ]:
# ─── 7a. ENCODER ────────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    """Bidirectional 2-layer LSTM encoder with dropout."""

    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, n_layers: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = n_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if n_layers > 1 else 0.0,
        )
        # Project bidir final state → decoder hidden dim
        self.fc_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, embed)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: (batch, src_len, hidden*2)
        # hidden, cell: (n_layers*2, batch, hidden)

        # Concatenate top-layer forward + backward final states
        h = torch.tanh(self.fc_h(torch.cat([hidden[-2], hidden[-1]], dim=1)))
        c = torch.tanh(self.fc_c(torch.cat([cell[-2],  cell[-1]],   dim=1)))
        return outputs, h, c  # outputs used by attention; h, c seed decoder

In [ ]:
# ─── 7b. BAHDANAU ATTENTION ─────────────────────────────────────────────────────

class BahdanauAttention(nn.Module):
    """
    Additive attention (Bahdanau et al., 2015).
    score(h_dec, h_enc_i) = v · tanh(W_dec·h_dec + W_enc·h_enc_i)
    """

    def __init__(self, enc_hid_dim: int, dec_hid_dim: int, attn_dim: int):
        super().__init__()
        self.W_enc = nn.Linear(enc_hid_dim, attn_dim, bias=False)
        self.W_dec = nn.Linear(dec_hid_dim, attn_dim, bias=False)
        self.v     = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor):
        # dec_hidden:  (batch, dec_hid)
        # enc_outputs: (batch, src_len, enc_hid*2)
        src_len = enc_outputs.shape[1]

        dec_exp = dec_hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, dec_hid)
        energy  = torch.tanh(self.W_enc(enc_outputs) + self.W_dec(dec_exp))  # (batch, src_len, attn_dim)
        scores  = self.v(energy).squeeze(2)                                   # (batch, src_len)

        attn_weights = F.softmax(scores, dim=1)                               # (batch, src_len)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)# (batch, enc_hid*2)
        return context, attn_weights

In [ ]:
# ─── 7c. DECODER ────────────────────────────────────────────────────────────────

class Decoder(nn.Module):
    """Single-direction LSTM decoder with Bahdanau attention."""

    def __init__(self, vocab_size: int, embed_dim: int, enc_hid_dim: int,
                 dec_hid_dim: int, attn_dim: int, n_layers: int, dropout: float):
        super().__init__()
        self.vocab_size = vocab_size
        self.attention  = BahdanauAttention(enc_hid_dim * 2, dec_hid_dim, attn_dim)
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = embed_dim + enc_hid_dim * 2,   # embed + context
            hidden_size = dec_hid_dim,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.fc_out = nn.Linear(dec_hid_dim + enc_hid_dim * 2 + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_token : torch.Tensor,   # (batch,)
        hidden      : torch.Tensor,   # (n_layers, batch, dec_hid)
        cell        : torch.Tensor,   # (n_layers, batch, dec_hid)
        enc_outputs : torch.Tensor,   # (batch, src_len, enc_hid*2)
    ):
        embedded = self.dropout(self.embedding(input_token.unsqueeze(1)))  # (batch, 1, embed)

        # Use top-layer hidden for attention query
        context, attn_w = self.attention(hidden[-1], enc_outputs)          # (batch, enc_hid*2)

        lstm_in = torch.cat([embedded, context.unsqueeze(1)], dim=2)       # (batch, 1, embed+enc_hid*2)
        output, (hidden, cell) = self.lstm(lstm_in, (hidden, cell))
        # output: (batch, 1, dec_hid)

        # Deep output: concatenate decoder output, context, embedding
        prediction = self.fc_out(
            torch.cat([output.squeeze(1), context, embedded.squeeze(1)], dim=1)
        )  # (batch, vocab_size)

        return prediction, hidden, cell, attn_w

In [ ]:
# ─── 7d. SEQ2SEQ WRAPPER ────────────────────────────────────────────────────────

class Seq2Seq(nn.Module):
    """Encoder–Decoder wrapper with teacher forcing."""

    def __init__(self, encoder: Encoder, decoder: Decoder, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(
        self,
        src: torch.Tensor,
        trg: torch.Tensor,
        teacher_forcing_ratio: float = 0.5,
    ) -> torch.Tensor:
        batch_size  = src.shape[0]
        trg_len     = trg.shape[1]
        vocab_size  = self.decoder.vocab_size

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)

        enc_out, hidden, cell = self.encoder(src)

        # Expand hidden/cell to match decoder n_layers
        n_dec_layers = self.decoder.lstm.num_layers
        hidden = hidden.unsqueeze(0).repeat(n_dec_layers, 1, 1)
        cell   = cell.unsqueeze(0).repeat(n_dec_layers, 1, 1)

        input_tok = trg[:, 0]   # <SOS>

        for t in range(1, trg_len):
            pred, hidden, cell, _ = self.decoder(input_tok, hidden, cell, enc_out)
            outputs[:, t, :] = pred
            use_teacher = random.random() < teacher_forcing_ratio
            input_tok   = trg[:, t] if use_teacher else pred.argmax(dim=1)

        return outputs  # (batch, trg_len, vocab_size)


# ─── Instantiate model ──────────────────────────────────────────────────────────
encoder = Encoder(
    vocab_size  = len(src_vocab),
    embed_dim   = mc['embed_dim'],
    hidden_dim  = mc['hidden_dim'],
    n_layers    = mc['n_layers'],
    dropout     = mc['dropout'],
).to(DEVICE)

decoder = Decoder(
    vocab_size   = len(trg_vocab),
    embed_dim    = mc['embed_dim'],
    enc_hid_dim  = mc['hidden_dim'],
    dec_hid_dim  = mc['hidden_dim'],
    attn_dim     = mc['attn_dim'],
    n_layers     = mc['n_layers'],
    dropout      = mc['dropout'],
).to(DEVICE)

model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

# Xavier uniform initialisation
def init_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight)
model.apply(init_weights)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 8. Training

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=tc['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=tc['lr_patience'], verbose=True
)


def run_epoch(
    model: Seq2Seq,
    loader: DataLoader,
    optimizer: Optional[torch.optim.Optimizer],
    criterion: nn.Module,
    teacher_forcing_ratio: float,
    clip: float,
    is_train: bool,
) -> float:
    model.train() if is_train else model.eval()
    total_loss = 0.0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            src = batch['src'].to(DEVICE)
            trg = batch['trg'].to(DEVICE)

            output = model(src, trg, teacher_forcing_ratio if is_train else 0.0)
            # output: (batch, trg_len, vocab_size)
            # Shift: ignore <SOS> position
            output_flat = output[:, 1:, :].reshape(-1, len(trg_vocab))
            target_flat = trg[:, 1:].reshape(-1)

            loss = criterion(output_flat, target_flat)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)


# ─── Training loop ──────────────────────────────────────────────────────────────
train_losses, dev_losses = [], []
best_dev_loss    = float('inf')
no_improve_count = 0

print(f"{'Epoch':>6} {'Train Loss':>12} {'Dev Loss':>10} {'Train PPL':>11} {'Dev PPL':>9} {'LR':>10}")
print('-' * 65)

for epoch in range(1, tc['epochs'] + 1):
    t0 = time.time()

    train_loss = run_epoch(model, train_loader, optimizer, criterion,
                           tc['teacher_forcing_ratio'], tc['clip_grad'], is_train=True)
    dev_loss   = run_epoch(model, dev_loader,   None,      criterion,
                           0.0,                   tc['clip_grad'], is_train=False)

    train_losses.append(train_loss)
    dev_losses.append(dev_loss)
    scheduler.step(dev_loss)

    current_lr = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - t0

    print(f"{epoch:>6} {train_loss:>12.4f} {dev_loss:>10.4f} "
          f"{math.exp(train_loss):>11.2f} {math.exp(dev_loss):>9.2f} {current_lr:>10.2e}  [{elapsed:.0f}s]")

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        no_improve_count = 0
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'dev_loss'   : best_dev_loss,
            'src_vocab'  : src_vocab,
            'trg_vocab'  : trg_vocab,
            'model_config': mc,
        }, tc['checkpoint_path'])
        print(f"  ✓ Saved best model (dev_loss={best_dev_loss:.4f})")
    else:
        no_improve_count += 1
        if no_improve_count >= tc['early_stop_patience']:
            print(f'Early stopping at epoch {epoch}.')
            break

In [ ]:
# ─── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(train_losses) + 1)

axes[0].plot(epochs_range, train_losses, label='Train', marker='o', markersize=3)
axes[0].plot(epochs_range, dev_losses,   label='Dev',   marker='o', markersize=3)
axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [math.exp(l) for l in train_losses], label='Train', marker='o', markersize=3)
axes[1].plot(epochs_range, [math.exp(l) for l in dev_losses],   label='Dev',   marker='o', markersize=3)
axes[1].set_title('Perplexity'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Inference — Greedy & Beam Search

In [ ]:
# Load best checkpoint
checkpoint = torch.load(tc['checkpoint_path'], map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (dev_loss={checkpoint['dev_loss']:.4f})")


@torch.no_grad()
def greedy_decode(model: Seq2Seq, src_tensor: torch.Tensor,
                  max_len: int) -> List[int]:
    """Single-sentence greedy decoding."""
    src  = src_tensor.unsqueeze(0).to(DEVICE)  # (1, src_len)
    enc_out, hidden, cell = model.encoder(src)

    n_dec = model.decoder.lstm.num_layers
    hidden = hidden.unsqueeze(0).repeat(n_dec, 1, 1)
    cell   = cell.unsqueeze(0).repeat(n_dec, 1, 1)

    input_tok = torch.tensor([trg_vocab.sos_idx], dtype=torch.long).to(DEVICE)
    generated = []

    for _ in range(max_len):
        pred, hidden, cell, _ = model.decoder(input_tok, hidden, cell, enc_out)
        top1 = pred.argmax(dim=1)
        if top1.item() == trg_vocab.eos_idx:
            break
        generated.append(top1.item())
        input_tok = top1

    return generated


@torch.no_grad()
def beam_search(
    model     : Seq2Seq,
    src_tensor: torch.Tensor,
    beam_size : int,
    max_len   : int,
    length_penalty: float,
) -> List[int]:
    """
    Beam search with length penalty.
    Returns the token ID list of the best hypothesis.
    """
    src = src_tensor.unsqueeze(0).to(DEVICE)
    enc_out, hidden, cell = model.encoder(src)

    n_dec  = model.decoder.lstm.num_layers
    hidden = hidden.unsqueeze(0).repeat(n_dec, 1, 1)
    cell   = cell.unsqueeze(0).repeat(n_dec, 1, 1)

    # Beam: list of (log_prob, token_ids, hidden, cell)
    beams = [(0.0, [trg_vocab.sos_idx], hidden, cell)]
    completed = []

    for _ in range(max_len):
        candidates = []
        for log_prob, tokens, h, c in beams:
            if tokens[-1] == trg_vocab.eos_idx:
                completed.append((log_prob, tokens))
                continue
            tok_t = torch.tensor([tokens[-1]], dtype=torch.long).to(DEVICE)
            pred, h_new, c_new, _ = model.decoder(tok_t, h, c, enc_out)
            log_probs = F.log_softmax(pred, dim=1).squeeze(0)   # (vocab,)
            topk_lp, topk_ids = log_probs.topk(beam_size)

            for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
                candidates.append((
                    log_prob + lp,
                    tokens + [tid],
                    h_new, c_new
                ))

        if not candidates:
            break

        # Length-normalised score
        def score(x):
            lp, toks, _, _ = x
            return lp / (len(toks) ** length_penalty)

        beams = sorted(candidates, key=score, reverse=True)[:beam_size]

    if not completed:
        completed = [(lp, toks) for lp, toks, _, _ in beams]

    best_lp, best_toks = max(
        completed,
        key=lambda x: x[0] / (len(x[1]) ** length_penalty)
    )
    # Strip SOS / EOS
    result = [t for t in best_toks
              if t not in (trg_vocab.sos_idx, trg_vocab.eos_idx)]
    return result


def translate_dataset(
    model       : Seq2Seq,
    df          : pd.DataFrame,
    src_vocab   : Vocabulary,
    trg_vocab   : Vocabulary,
    use_beam    : bool = True,
    beam_size   : int  = 5,
    max_len     : int  = 80,
    length_penalty: float = 0.7,
) -> Tuple[List[str], List[float]]:
    """Translate all rows in df and return (predictions, per_sentence_inference_times)."""
    model.eval()
    predictions, times = [], []

    for _, row in df.iterrows():
        src_ids = src_vocab.encode(row['src_tokens'], add_sos=False, add_eos=True)
        src_t   = torch.tensor(src_ids, dtype=torch.long)

        t0 = time.perf_counter()
        if use_beam:
            token_ids = beam_search(model, src_t, beam_size, max_len, length_penalty)
        else:
            token_ids = greedy_decode(model, src_t, max_len)
        times.append(time.perf_counter() - t0)

        predictions.append(trg_vocab.decode(token_ids))

    return predictions, times


print('Inference functions defined.')

## 10. Evaluation: BLEU · BERTScore · Efficiency

In [ ]:
def compute_bleu(predictions: List[str], references: List[str]) -> float:
    """
    NLTK corpus_bleu with NO weights (uniform 1-gram to 4-gram)
    as specified in Assignment 2.
    """
    refs  = [[ref.split()] for ref in references]
    hyps  = [pred.split() for pred in predictions]
    # weights=None → default uniform weights in nltk corpus_bleu
    bleu = corpus_bleu(refs, hyps)
    return bleu * 100  # return as percentage


def compute_bert_score(predictions: List[str], references: List[str]) -> float:
    """
    BERTScore F1 with rescale_with_baseline=True.
    Uses microsoft/deberta-xlarge-mnli by default (bert_score library default).
    """
    P, R, F1 = bert_score_fn(
        predictions,
        references,
        lang='en',
        rescale_with_baseline=True,
        verbose=False,
    )
    return F1.mean().item() * 100  # percentage


def evaluate_split(
    model       : Seq2Seq,
    df          : pd.DataFrame,
    src_vocab   : Vocabulary,
    trg_vocab   : Vocabulary,
    split_name  : str,
    use_beam    : bool = True,
    ic          : dict = INFER_CONFIG,
) -> Dict:
    """Run full evaluation on one split: BLEU, BERTScore, timing, param count."""
    print(f'\n── Evaluating on {split_name} set ──')

    # ── Inference ────────────────────────────────────────────────────────────────
    t_start = time.perf_counter()
    predictions, per_sent_times = translate_dataset(
        model, df, src_vocab, trg_vocab,
        use_beam=use_beam,
        beam_size=ic['beam_size'],
        max_len=ic['max_decode_len'],
        length_penalty=ic['length_penalty'],
    )
    total_inference_time = time.perf_counter() - t_start

    # ── Metrics ──────────────────────────────────────────────────────────────────
    references = df['english'].str.lower().tolist()
    bleu       = compute_bleu(predictions, references)
    bert_f1    = compute_bert_score(predictions, references)

    # ── Efficiency ───────────────────────────────────────────────────────────────
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'  BLEU                  : {bleu:.2f}')
    print(f'  BERTScore F1          : {bert_f1:.2f}')
    print(f'  Total inference time  : {total_inference_time:.2f}s ({len(df)} sentences)')
    print(f'  Avg per sentence      : {total_inference_time/len(df)*1000:.1f}ms')
    print(f'  Total model parameters: {n_params:,}')

    return {
        'split'           : split_name,
        'predictions'     : predictions,
        'references'      : references,
        'bleu'            : bleu,
        'bert_f1'         : bert_f1,
        'total_time_s'    : total_inference_time,
        'n_params'        : n_params,
    }


# ─── Evaluate dev set ───────────────────────────────────────────────────────────
dev_results = evaluate_split(model, dev_df, src_vocab, trg_vocab, 'dev', use_beam=True)

## 11. Generate submission.csv (Test Set)

In [ ]:
print('Generating test set translations...')
test_preds, test_times = translate_dataset(
    model, test_df, src_vocab, trg_vocab,
    use_beam=True,
    beam_size=INFER_CONFIG['beam_size'],
    max_len=INFER_CONFIG['max_decode_len'],
    length_penalty=INFER_CONFIG['length_penalty'],
)

total_test_time = sum(test_times)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total inference time (test): {total_test_time:.2f}s')
print(f'Total model parameters     : {n_params:,}')

# Build submission.csv — format per assignment: Source_id, Sentence_en
submission = pd.DataFrame({
    DATA_CONFIG['id_col']: test_df[DATA_CONFIG['id_col']].tolist(),
    'Sentence_en'        : test_preds,
})
submission.to_csv('submission.csv', index=False, encoding='utf-8')
print('\nsubmission.csv saved.')
print(submission.head(10))

## 12. Translation Examples & Error Analysis

In [ ]:
def show_translation_examples(
    df          : pd.DataFrame,
    predictions : List[str],
    n           : int = 10,
    random_sample: bool = True,
):
    """Display n examples with source, reference, prediction, and per-sentence BLEU."""
    indices = random.sample(range(len(df)), min(n, len(df))) if random_sample else list(range(n))

    smooth = SmoothingFunction().method1
    print(f"{'='*90}")
    for i, idx in enumerate(indices):
        src_text  = df.iloc[idx]['sanskrit']
        ref_text  = df.iloc[idx]['english'].lower()
        pred_text = predictions[idx]

        # Per-sentence BLEU with smoothing
        from nltk.translate.bleu_score import sentence_bleu
        s_bleu = sentence_bleu([ref_text.split()], pred_text.split(),
                               smoothing_function=smooth) * 100

        print(f"Example {i+1}")
        print(f"  Source    : {src_text}")
        print(f"  Reference : {ref_text}")
        print(f"  Prediction: {pred_text}")
        print(f"  BLEU      : {s_bleu:.1f}")

        # Simple error categorisation
        ref_toks  = set(ref_text.split())
        pred_toks = set(pred_text.split())
        overlap   = len(ref_toks & pred_toks) / max(len(ref_toks), 1)
        if s_bleu > 50:
            error_cat = 'Good translation'
        elif overlap > 0.5:
            error_cat = 'Lexically close but reordering issues'
        elif len(pred_text.split()) < len(ref_text.split()) * 0.5:
            error_cat = 'Under-generation (too short)'
        elif len(pred_text.split()) > len(ref_text.split()) * 1.8:
            error_cat = 'Over-generation (too long / repetition)'
        else:
            error_cat = 'Lexical mismatch'
        print(f"  Error cat : {error_cat}")
        print(f"{'─'*90}")


show_translation_examples(dev_df, dev_results['predictions'], n=10)

In [ ]:
# ─── Attention alignment visualisation ─────────────────────────────────────────
@torch.no_grad()
def get_attention_weights(
    model     : Seq2Seq,
    src_tokens: List[str],
    src_vocab : Vocabulary,
    trg_vocab : Vocabulary,
    max_len   : int = 50,
) -> Tuple[List[str], List[List[float]]]:
    """Returns (generated_tokens, attention_matrix)."""
    model.eval()
    src_ids = src_vocab.encode(src_tokens, add_sos=False, add_eos=True)
    src_t   = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)

    enc_out, hidden, cell = model.encoder(src_t)
    n_dec  = model.decoder.lstm.num_layers
    hidden = hidden.unsqueeze(0).repeat(n_dec, 1, 1)
    cell   = cell.unsqueeze(0).repeat(n_dec, 1, 1)

    input_tok = torch.tensor([trg_vocab.sos_idx], dtype=torch.long).to(DEVICE)
    generated, attn_matrix = [], []

    for _ in range(max_len):
        pred, hidden, cell, attn_w = model.decoder(input_tok, hidden, cell, enc_out)
        top1 = pred.argmax(dim=1)
        if top1.item() == trg_vocab.eos_idx:
            break
        generated.append(trg_vocab.idx2token[top1.item()])
        attn_matrix.append(attn_w.squeeze(0).cpu().tolist())
        input_tok = top1

    return generated, attn_matrix


def plot_attention(src_tokens, trg_tokens, attn_matrix, title='Attention Alignment'):
    fig, ax = plt.subplots(figsize=(max(6, len(src_tokens)*0.7),
                                    max(4, len(trg_tokens)*0.5)))
    matrix = np.array(attn_matrix)   # (trg_len, src_len)
    im = ax.imshow(matrix, cmap='YlOrBr', aspect='auto')
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_yticklabels(trg_tokens, fontsize=9)
    ax.set_xlabel('Sanskrit source', fontsize=10)
    ax.set_ylabel('English output',  fontsize=10)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.savefig('attention_alignment.png', dpi=150, bbox_inches='tight')
    plt.show()


# Visualise attention for one dev example
sample_row = dev_df.iloc[0]
gen_tokens, attn_mat = get_attention_weights(
    model, sample_row['src_tokens'], src_vocab, trg_vocab
)
src_display = sample_row['src_tokens'][:len(attn_mat[0])] if attn_mat else sample_row['src_tokens']
plot_attention(src_display, gen_tokens, attn_mat,
               title=f"Attention: {sample_row['sanskrit'][:60]}")

## 13. Final Evaluation Summary

In [ ]:
print('='*60)
print('FINAL EVALUATION SUMMARY')
print('='*60)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nDev set:")
print(f"  BLEU              : {dev_results['bleu']:.2f}")
print(f"  BERTScore F1      : {dev_results['bert_f1']:.2f}")
print(f"  Inference time    : {dev_results['total_time_s']:.2f}s")

print(f"\nModel:")
print(f"  Total parameters  : {n_params:,}")
print(f"  Embed dim         : {mc['embed_dim']}")
print(f"  Hidden dim        : {mc['hidden_dim']} (per direction)")
print(f"  Encoder           : BiLSTM {mc['n_layers']}-layer")
print(f"  Attention         : Bahdanau (attn_dim={mc['attn_dim']})")
print(f"  Decoder           : LSTM {mc['n_layers']}-layer")
print(f"  Beam size         : {INFER_CONFIG['beam_size']}")

print(f"\nFiles generated:")
print(f"  submission.csv        — test set predictions")
print(f"  best_model.pt         — model checkpoint")
print(f"  training_curves.png   — loss/perplexity plot")
print(f"  attention_alignment.png — sample attention heatmap")
print('='*60)